Project Title:
Neuro-Adaptive Early Intervention System: Predicting ASD Traits and Recommending Personalized Learning Paths

Summary of Achievements:
Architected an end-to-end Neuro-Adaptive Early Intervention System that integrates Agentic AI via the google-genai SDK to predict ASD traits and automate specialized support. Developed a high-precision machine learning pipeline for trait detection, coupled with a Generative AI Agent that dynamically synthesizes behavioral telemetry into personalized learning paths. By utilizing the reasoning capabilities of Google’s Gemini models, the system moves beyond static diagnostics to provide real-time, individualized educational roadmaps. This innovation streamlines the diagnostic-to-intervention lifecycle, transforming complex neuro-developmental data into a proactive tool for clinical and educational empowerment.



In [1]:
!C:\ProgramData\anaconda3\python.exe -m pip install --upgrade typing-extensions

Defaulting to user installation because normal site-packages is not writeable
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)


In [1]:
import kagglehub
import pandas as pd
import os

print("--- FETCHING CLEAN TODDLER ASD DATASET ---")

# 1. Download the verified Toddler ASD dataset
path = kagglehub.dataset_download("fabdelja/autism-screening-for-toddlers")

# 2. Find the CSV file inside the downloaded folder
csv_files =[f for f in os.listdir(path) if f.endswith('.csv')]
if csv_files:
    exact_file_path = os.path.join(path, csv_files[0])
    print(f"Found clean dataset: {csv_files[0]}")
    
    # 3. Load it into Pandas!
    df = pd.read_csv(exact_file_path)
    
    print(f"\nDataset Shape: {df.shape[0]} Toddlers, {df.shape[1]} Features")
    display(df.head())
else:
    print("Error: No CSV file found in the downloaded folder.")

--- FETCHING CLEAN TODDLER ASD DATASET ---
Found clean dataset: Autism_Screening_Data_Combined.csv

Dataset Shape: 6075 Toddlers, 15 Features


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,Age,Sex,Jauundice,Family_ASD,Class
0,1,1,0,1,0,0,1,1,0,0,15,m,no,no,NO
1,0,1,1,1,0,1,1,0,1,0,15,m,no,no,NO
2,1,1,1,0,1,1,1,1,1,1,15,f,no,yes,YES
3,1,1,1,1,1,1,1,1,0,0,16,f,no,no,YES
4,1,1,1,1,1,1,1,1,1,1,15,f,no,no,YES


Building a Content-Based Filtering System powered by NLP. Building the Neuro-Adaptive Early Intervention System.

Step 1: Clean the Data & Train the Predictor

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("--- PHASE 1: BUILDING THE ASD PREDICTOR ---")

# 1. Look at the columns to understand the data
print("\nOriginal Columns:", df.columns.tolist())

# 2. Identify the Target Variable
# The target column is usually the last one, named something like 'Class/ASD Traits ' or 'Class/ASD'
# Let's find it dynamically
target_col =[col for col in df.columns if 'Class' in col or 'ASD' in col][-1]
print(f"Target Column Identified: '{target_col}'")

# 3. Clean the Target Variable (Convert "Yes"/"No" to 1/0)
# If it's already 1/0, this won't break anything
if df[target_col].dtype == 'object':
    df['Target'] = df[target_col].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)
else:
    df['Target'] = df[target_col]

# 4. Select Features (The 10 Behavioral Questions + Age + Sex if available)
# We grab columns A1 through A10
feature_cols =[col for col in df.columns if col.startswith('A') and col[1:].isdigit()]

# Add other useful features if they exist (e.g., 'Age_Mons', 'Sex')
optional_features = ['Age_Mons', 'Sex']
for col in optional_features:
    if col in df.columns:
        feature_cols.append(col)

print(f"\nFeatures Selected for Training: {feature_cols}")

# 5. Clean Categorical Features (like 'Sex' if it exists)
X = df[feature_cols].copy()
if 'Sex' in X.columns:
    X['Sex'] = X['Sex'].apply(lambda x: 1 if str(x).strip().lower() in ['m', 'males', 'male'] else 0) # 1 for Male, 0 for Female

y = df['Target']

# 6. Train the XGBoost Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining XGBoost Classifier...")
asd_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
asd_model.fit(X_train, y_train)

# 7. Evaluate the Model
predictions = asd_model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\n✅ Model Training Complete!")
print(f"Model Accuracy on Test Data: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, predictions))

--- PHASE 1: BUILDING THE ASD PREDICTOR ---

Original Columns: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'Age', 'Sex', 'Jauundice', 'Family_ASD', 'Class']
Target Column Identified: 'Class'

Features Selected for Training: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'Sex']

Training XGBoost Classifier...

✅ Model Training Complete!
Model Accuracy on Test Data: 93.83%

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.94      0.96       848
           1       0.87      0.93      0.90       367

    accuracy                           0.94      1215
   macro avg       0.92      0.94      0.93      1215
weighted avg       0.94      0.94      0.94      1215



The Live Scraper (Phase 2): Runs the google-play-scraper to search the live Android store for "autism speech therapy special education," and creates the df_apps database containing the top 7 to 15 apps and their descriptions.

In [3]:
pip install google-play-scraper pandas

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [4]:
from google_play_scraper import search, app
import pandas as pd

print("--- SEARCHING PLAY STORE FOR LIVE AUTISM/SPEECH APPS ---")

# 1. Dynamically search the Play Store for apps related to "Autism" and "Speech Delay"
search_results = search(
    "autism speech therapy special education",
    lang="en",
    country="us"
)

# Grab the top 15 apps from the search results
top_apps = search_results[:15]
print(f"Found {len(top_apps)} relevant apps. Fetching detailed data...")

app_data =[]

# 2. Loop through the dynamically found apps and pull their detailed data
for result in top_apps:
    pkg_id = result['appId'] # We get the live, working ID directly from the search!
    try:
        # Get the full app details
        app_details = app(pkg_id, lang='en', country='us')
        
        # We only want educational/medical apps, not random games that show up in search
        if app_details['genre'] in ['Education', 'Medical', 'Parenting', 'Health & Fitness']:
            app_data.append({
                'App_Name': app_details['title'],
                'Category': app_details['genre'],
                'Rating': round(app_details.get('score', 0), 2),
                'Total_Reviews': app_details.get('reviews', 0),
                'Installs': app_details.get('installs', 'Unknown'),
                'Price': "Free" if app_details.get('free') else "Paid",
                'Description': app_details['description'][:500] # Grabbing more text for better NLP matching
            })
            print(f"✅ Secured data for: {app_details['title']}")
            
    except Exception as e:
        print(f"❌ Failed to fetch details for {pkg_id}: {e}")

# Convert to a DataFrame
df_apps = pd.DataFrame(app_data)

print(f"\n--- LIVE DATABASE READY WITH {len(df_apps)} APPS ---")
display(df_apps.head())

--- SEARCHING PLAY STORE FOR LIVE AUTISM/SPEECH APPS ---
Found 15 relevant apps. Fetching detailed data...
✅ Secured data for: Autism Speech and Language
✅ Secured data for: Speech Blubs: Language Therapy
✅ Secured data for: Language Therapy for Children 
✅ Secured data for: BASICS: Speech | Autism | ADHD
✅ Secured data for: SpeakEasy: Home Speech Therapy
✅ Secured data for: Speech Therapy: Let Me Talk
✅ Secured data for: Autism Soundboard
✅ Secured data for: Speech Therapy 1 – Preverbal
✅ Secured data for: Speech Therapy 3 – Learn Words

--- LIVE DATABASE READY WITH 9 APPS ---


,App_Name,Category,Rating,Total_Reviews,Installs,Price,Description
0,Autism Speech and Language,Parenting,4.02,16,"10,000+",Free,Autism Speech and Language brings you the Auti...
1,Speech Blubs: Language Therapy,Parenting,4.46,1606,"5,000,000+",Free,Do you need more proof? Check out the featured...
2,Language Therapy for Children,Education,4.65,205,"1,000,000+",Free,• THE FIRST AND ONLY LANGUAGE THERAPY APPLICAT...
3,BASICS: Speech | Autism | ADHD,Education,3.94,10,"500,000+",Free,BASICS is an award-winning early learning app ...
4,SpeakEasy: Home Speech Therapy,Parenting,3.64,159,"100,000+",Free,<h1><b>SpeakEasy has been shown by a rigorous ...


The Recommendation Engine (Phase 3): It takes the test_toddler profile, uses the asd_model from Cell 1 to predict the risk, translates the toddler's needs into a text string, and then uses NLP (TfidfVectorizer) to match those needs against the df_apps database from Cell 2.


Step 2: Connecting the Live Data to the Recommender

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("==========================================================")
print("   LIVE NEURO-ADAPTIVE LEARNING RECOMMENDATION SYSTEM     ")
print("==========================================================\n")

# =========================================================
# 1. DEFINING THE TODDLER'S PROFILE (The "Test Patient")
# =========================================================
# We use the same 30-month-old toddler profile
test_toddler = pd.DataFrame({
    'A1': [1], 'A2': [0], 'A3': [1], 'A4': [1], 'A5': [0], 
    'A6': [1], 'A7': [1], 'A8': [0], 'A9': [1], 'A10':[1], 
    'Sex': [1] # Make sure this matches the capitalization in your training data!
})

# =========================================================
# 2. PREDICTING ASD RISK
# =========================================================
print(">>> ANALYZING TODDLER BEHAVIORAL PROFILE...")
risk_probability = asd_model.predict_proba(test_toddler)[0][1] * 100
print(f"    [RESULT] ASD Trait Probability: {risk_probability:.2f}%\n")

if risk_probability >= 50:
    print("    [ALERT] High likelihood of ASD traits detected. Searching LIVE database...\n")
    
    # =========================================================
    # 3. BUILDING THE RECOMMENDATION ENGINE (Using Scraped Data)
    # =========================================================
    
    # Step A: Translate the numeric scores into text "Needs"
    toddler_needs = []
    if test_toddler['A1'].iloc[0] == 1 or test_toddler['A7'].iloc[0] == 1:
        toddler_needs.append("speech delay non-verbal communication talk words")
    if test_toddler['A3'].iloc[0] == 1 or test_toddler['A4'].iloc[0] == 1:
        toddler_needs.append("social interaction play cognitive learning")
    if test_toddler['A9'].iloc[0] == 1 or test_toddler['A10'].iloc[0] == 1:
        toddler_needs.append("sensory meltdowns routine calm visual")
        
    toddler_profile_text = " ".join(toddler_needs)
    
    # Step B: Use TF-IDF to find the best matching LIVE resources
    tfidf = TfidfVectorizer(stop_words='english')
    
    # We use the 'Description' column you scraped from the Play Store!
    # Fill any missing descriptions with empty text to avoid errors
    app_descriptions = df_apps['Description'].fillna("").tolist()
    
    tfidf_matrix = tfidf.fit_transform(app_descriptions)
    toddler_vector = tfidf.transform([toddler_profile_text])
    
    # Calculate Cosine Similarity
    similarity_scores = cosine_similarity(toddler_vector, tfidf_matrix).flatten()
    
    # Add scores back to your live scraped dataframe
    df_apps['Match_Score_%'] = (similarity_scores * 100).round(1)
    
    # Sort to find the best apps!
    top_recommendations = df_apps.sort_values(by='Match_Score_%', ascending=False).head(3)
    
    # =========================================================
    # 4. FINAL OUTPUT
    # =========================================================
    print("==========================================================")
    print("   PERSONALIZED LEARNING INTERVENTION PLAN (LIVE DATA)   ")
    print("==========================================================")
    for index, row in top_recommendations.iterrows():
       print(f"⭐ {row['App_Name']} (Match: {row['Match_Score_%']}%)")
       print(f"   Category: {row['Category']} | Rating: {row['Rating']}⭐")
    print(f"   Price: {row['Price']} | Installs: {row['Installs']}")
        # Print a short snippet of why it matched
    print(f"   Description Snippet: {row['Description'][:100]}...\n")

else:
    print("[RESULT] Low likelihood of ASD traits. Standard milestones observed.")

   LIVE NEURO-ADAPTIVE LEARNING RECOMMENDATION SYSTEM     

>>> ANALYZING TODDLER BEHAVIORAL PROFILE...
    [RESULT] ASD Trait Probability: 39.27%

[RESULT] Low likelihood of ASD traits. Standard milestones observed.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("==========================================================")
print("   LIVE NEURO-ADAPTIVE LEARNING RECOMMENDATION SYSTEM     ")
print("==========================================================\n")

# =========================================================
# 1. DEFINING THE TODDLER'S PROFILE (The "Test Patient")
# =========================================================
# A score of 1 means the child failed the behavioral milestone.
# We are creating a profile with severe communication and sensory delays.
test_toddler = pd.DataFrame({
    'A1': [1],  # Does not look when called
    'A2': [1],  # Does not make eye contact 
    'A3': [1],  # Does not point to indicate interest 
    'A4': [1],  # Does not point to share interest 
    'A5': [1],  # Does not pretend play
    'A6': [1],  # Does not follow where you look 
    'A7': [1],  # No words / Severe Speech delay
    'A8': [1],  # Does not understand simple gestures
    'A9': [1],  # Sensory processing differences (Stares at nothing)
    'A10':[1],  # Flaps hands / Meltdowns
    'Sex': [1]  # Male
})

# =========================================================
# 2. PREDICTING ASD RISK
# =========================================================
print(">>> ANALYZING TODDLER BEHAVIORAL PROFILE...")
risk_probability = asd_model.predict_proba(test_toddler)[0][1] * 100
print(f"    [RESULT] ASD Trait Probability: {risk_probability:.2f}%\n")

if risk_probability >= 50:
    print("    [ALERT] High likelihood of ASD traits detected. Searching LIVE database...\n")
    
    # =========================================================
    # 3. BUILDING THE RECOMMENDATION ENGINE (Using Scraped Data)
    # =========================================================
    
    # Step A: Translate the numeric scores into text "Needs"
    toddler_needs = []
    if test_toddler['A1'].iloc[0] == 1 or test_toddler['A7'].iloc[0] == 1:
        toddler_needs.append("speech delay non-verbal communication talk words")
    if test_toddler['A3'].iloc[0] == 1 or test_toddler['A4'].iloc[0] == 1:
        toddler_needs.append("social interaction play cognitive learning")
    if test_toddler['A9'].iloc[0] == 1 or test_toddler['A10'].iloc[0] == 1:
        toddler_needs.append("sensory meltdowns routine calm visual")
        
    toddler_profile_text = " ".join(toddler_needs)
    
    # Step B: Use TF-IDF to find the best matching LIVE resources
    tfidf = TfidfVectorizer(stop_words='english')
    
    # We use the 'Description' column you scraped from the Play Store!
    # Fill any missing descriptions with empty text to avoid errors
    app_descriptions = df_apps['Description'].fillna("").tolist()
    
    tfidf_matrix = tfidf.fit_transform(app_descriptions)
    toddler_vector = tfidf.transform([toddler_profile_text])
    
    # Calculate Cosine Similarity
    similarity_scores = cosine_similarity(toddler_vector, tfidf_matrix).flatten()
    
    # Add scores back to your live scraped dataframe
    df_apps['Match_Score_%'] = (similarity_scores * 100).round(1)
    
    # Sort to find the best apps!
    top_recommendations = df_apps.sort_values(by='Match_Score_%', ascending=False).head(3)
    
    # =========================================================
    # 4. FINAL OUTPUT
    # =========================================================
    print("==========================================================")
    print("   PERSONALIZED LEARNING INTERVENTION PLAN (LIVE DATA)   ")
    print("==========================================================")
    for index, row in top_recommendations.iterrows():
        print(f"⭐ {row['App_Name']} (Match: {row['Match_Score_%']}%)")
        print(f"   Category: {row['Category']} | Rating: {row['Rating']}⭐")
        print(f"   Price: {row['Price']} | Installs: {row['Installs']}")
        # Print a short snippet of why it matched
        print(f"   Description Snippet: {row['Description'][:100]}...\n")

else:
    print("[RESULT] Low likelihood of ASD traits. Standard milestones observed.")

   LIVE NEURO-ADAPTIVE LEARNING RECOMMENDATION SYSTEM     

>>> ANALYZING TODDLER BEHAVIORAL PROFILE...
    [RESULT] ASD Trait Probability: 67.54%

    [ALERT] High likelihood of ASD traits detected. Searching LIVE database...

   PERSONALIZED LEARNING INTERVENTION PLAN (LIVE DATA)   
⭐ SpeakEasy: Home Speech Therapy (Match: 23.6%)
   Category: Parenting | Rating: 3.64⭐
   Price: Free | Installs: 100,000+
   Description Snippet: <h1><b>SpeakEasy has been shown by a rigorous randomized control trial to improve parent-child commu...

⭐ BASICS: Speech | Autism | ADHD (Match: 19.9%)
   Category: Education | Rating: 3.94⭐
   Price: Free | Installs: 500,000+
   Description Snippet: BASICS is an award-winning early learning app trusted by over 7 lakh families worldwide. Created by ...

⭐ Autism Soundboard (Match: 17.0%)
   Category: Education | Rating: 0.0⭐
   Price: Free | Installs: 1,000+
   Description Snippet: 💬 Autism Soundboard: AAC Communication App for Kids
The Autism Soundboard is th

Save Your ASD Model (In your Jupyter Notebook)

In [8]:
import joblib
# Assuming your trained model is named 'asd_model'
joblib.dump(asd_model, 'asd_model.pkl')
print("✅ Model saved successfully as 'asd_model.pkl'")

✅ Model saved successfully as 'asd_model.pkl'
